# EDA viewer — Barbados Historic Handwriting Challenge

**This notebook only *displays* results.** All analysis lives in `src/eda.py`,
`src/data/multiline.py` and `src/data/exclusions.py` so that it is reproducible
from one command and reviewable as code. Run these first:

```bash
sbatch slurm/01_prepare_data.sbatch
# or locally:
python -m src.eda --train_csv data/Train.csv --image_dir data/images --out_dir report/eda
```

No numbers are hardcoded here — every cell reads the generated CSVs.


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
EDA = "../report/eda"; DER = "../data/derived"
print("eda outputs present:", os.path.isdir(EDA))

## 1. Length distribution and — crucially — metric weight share

The metric weights longer references more heavily, so `weight_share` is what actually moves the leaderboard, **not** `row_share`.

In [ ]:
b = pd.read_csv(f"{EDA}/length_buckets.csv")
display(b)
ax = b.plot(x="bucket", y=["row_share","weight_share"], kind="bar", figsize=(8,4))
ax.set_title("Row share vs share of total metric weight"); plt.tight_layout(); plt.show()

## 2. Character inventory — the CER long tail

In [ ]:
c = pd.read_csv(f"{EDA}/charset.csv")
print("distinct characters:", len(c))
display(c.head(30))
print("\ncharacters appearing <= 5 times (archaic marks, ligatures, stray glyphs):")
display(c[c["count"] <= 5])

## 3. Archaic / structural markers

In [ ]:
display(pd.read_csv(f"{EDA}/archaic_markers.csv"))

## 4. Image geometry and aspect ratios

Drives the choice of `target_height` / `max_width` / padding regime.

In [ ]:
g = pd.read_csv(f"{EDA}/image_geometry.csv")
display(g[["width","height","aspect"]].describe())
g["aspect"].hist(bins=50, figsize=(7,4)); plt.xlabel("width/height"); plt.show()

## 5. Multi-line census — the unresolved label-convention issue

In [ ]:
m = pd.read_csv(f"{DER}/multiline_census.csv")
print("multi-line rate:", m["is_multiline"].mean())
display(m.loc[m["is_multiline"], "label_convention"].value_counts())
print("\nMAJORITY convention above is what training is biased toward.")
print("The minority is handled by label-noise-tolerant training, NOT by editing labels.")

## 6. Corrupted-row audit

In [ ]:
a = pd.read_csv("../report/exclusion_audit.csv")
print("excluded:", int(a["excluded"].sum()), "of", len(a))
print("organizer-flagged:", int(a["flagged_by_organizers"].sum()),
      "| auto-confirmed:", int(a["confirmed"].sum()))
display(a[a["flagged_by_organizers"]][["ID","confirmed","missing_image","blank_image",
                                        "empty_label","short_label","length_outlier"]])

## 7. Duplicates and near-duplicates

These must not straddle folds, or CV leaks.

In [ ]:
d = pd.read_csv(f"{EDA}/duplicate_texts.csv")
print("rows sharing an identical transcription:", len(d))
display(d.head(20))

## 8. Fold sanity

Every fold must have a comparable long-line population, or OOF comparisons are noise.

In [ ]:
f = pd.read_csv(f"{DER}/train_folds.csv")
display(f.groupby("fold")["label_len"].describe()[["count","mean","50%","max"]])
display(pd.crosstab(f["fold"], f["stratum"]))